# 🚀 Train model từ VIDEO THẬT của bạn → TFLite (tự động)

**Bạn chỉ cần:** quay vài video → upload → bấm Run All → tải `.tflite` về.

Notebook tự: **cắt frame → dò mặt → crop mắt/miệng → train CNN → xuất TFLite**.

## ⚠️ Quy tắc đặt TÊN VIDEO (để tự gán nhãn)
| Chế độ | Tên file chứa | → Nhãn |
|--------|---------------|--------|
| **eye** | `...open...` | eyes_open |
| | `...closed...` | eyes_closed |
| **yawn** | `...yawn...` | yawn |
| | `...noyawn...` / `...no_yawn...` | no_yawn |

Ví dụ: quay 1 video nhắm mắt → đặt tên `closed_1.mp4`; quay video mở mắt → `open_1.mp4`.

`Runtime → T4 GPU (hoặc CPU cũng được) → Run all`

In [ ]:
# 1 — Setup + tải model dò mặt + hàm crop
import os, cv2, numpy as np, urllib.request, shutil, math
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mpp
from mediapipe.tasks.python import vision

M = 'face_landmarker.task'
if not os.path.exists(M):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task', M)

_opts = vision.FaceLandmarkerOptions(base_options=mpp.BaseOptions(model_asset_path=M),
                                     running_mode=vision.RunningMode.IMAGE, num_faces=1)
LMK = vision.FaceLandmarker.create_from_options(_opts)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 0, 17, 13, 14]

def _box(img, pts, pad=0.5, size=64):
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
    x1,x2,y1,y2=min(xs),max(xs),min(ys),max(ys)
    w,h=x2-x1,y2-y1
    x1,x2=int(x1-w*pad),int(x2+w*pad); y1,y2=int(y1-h*pad),int(y2+h*pad)
    x1,y1=max(0,x1),max(0,y1)
    c=img[y1:y2, x1:x2]
    return cv2.resize(c,(size,size)) if c.size else None

def crops_from_face(img, mode):
    """Trả list ảnh crop 64x64. eye→2 mắt; yawn→1 miệng."""
    h,w=img.shape[:2]
    res=LMK.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks: return []
    lm=res.face_landmarks[0]; P=lambda i:(lm[i].x*w, lm[i].y*h)
    out=[]
    if mode=='eye':
        for idxs in (LEFT_EYE, RIGHT_EYE):
            c=_box(img,[P(i) for i in idxs]); 
            if c is not None: out.append(c)
    else:
        c=_box(img,[P(i) for i in MOUTH], pad=0.6)
        if c is not None: out.append(c)
    return out
print('✅ Sẵn sàng.')

In [ ]:
# 2 — CHỌN CHẾ ĐỘ + upload video
MODE = 'eye'      # 'eye' (mắt) hoặc 'yawn' (ngáp)
FPS  = 3          # số frame cắt mỗi giây

CLASSES = ['eyes_closed','eyes_open'] if MODE=='eye' else ['no_yawn','yawn']

def label_of(name):
    n=name.lower()
    if MODE=='eye':
        if 'closed' in n or 'close' in n: return 'eyes_closed'
        if 'open' in n: return 'eyes_open'
    else:
        if 'noyawn' in n or 'no_yawn' in n or 'no-yawn' in n: return 'no_yawn'
        if 'yawn' in n: return 'yawn'
    return None

from google.colab import files
print(f'CHẾ ĐỘ = {MODE} | classes = {CLASSES}')
print('Upload video (tên file chứa từ khóa class, xem bảng ở Cell 1):')
UP = files.upload()
for fn in UP:
    print(f'  {fn} → nhãn: {label_of(fn)}')

In [ ]:
# 3 — Cắt frame + crop ROI + chia train/val (tự động)
DATA='/content/realdata'
shutil.rmtree(DATA, ignore_errors=True)
for sp in ('train','val'):
    for c in CLASSES: os.makedirs(f'{DATA}/{sp}/{c}', exist_ok=True)

import random; random.seed(42)
counts={c:0 for c in CLASSES}
for fn in UP:
    lbl=label_of(fn)
    if lbl is None:
        print(f'⏭️  Bỏ qua {fn} (tên không chứa class)'); continue
    cap=cv2.VideoCapture(fn)
    native=cap.get(cv2.CAP_PROP_FPS) or 30
    step=max(1,round(native/FPS))
    idx=saved=0
    while True:
        ok,frame=cap.read()
        if not ok: break
        if idx%step==0:
            for j,crop in enumerate(crops_from_face(frame, MODE)):
                sp='val' if random.random()<0.2 else 'train'
                cv2.imwrite(f'{DATA}/{sp}/{lbl}/{Path(fn).stem}_{saved}_{j}.jpg', crop)
                counts[lbl]+=1; saved+=1
        idx+=1
    cap.release()
    print(f'✅ {fn}: crop {saved} ảnh ({lbl})')

print('\n📊 Tổng mỗi class:', counts)
if min(counts.values())<30:
    print('⚠️  Một class < 30 ảnh — quay thêm video hoặc tăng FPS để model học tốt hơn.')

In [ ]:
# 4 — Train CNN 64x64 trên data thật
import tensorflow as tf
from tensorflow.keras import layers, models

tr = tf.keras.utils.image_dataset_from_directory(f'{DATA}/train', image_size=(64,64),
        batch_size=32, label_mode='categorical', class_names=CLASSES)
va = tf.keras.utils.image_dataset_from_directory(f'{DATA}/val', image_size=(64,64),
        batch_size=32, label_mode='categorical', class_names=CLASSES)
norm = layers.Rescaling(1./255)
tr = tr.map(lambda x,y:(norm(x),y)).cache().prefetch(2)
va = va.map(lambda x,y:(norm(x),y)).cache().prefetch(2)

model = models.Sequential([
    layers.Input((64,64,3)),
    layers.Conv2D(32,3,activation='relu',padding='same'), layers.BatchNormalization(), layers.MaxPool2D(),
    layers.Conv2D(64,3,activation='relu',padding='same'), layers.BatchNormalization(), layers.MaxPool2D(),
    layers.Conv2D(128,3,activation='relu',padding='same'), layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'), layers.Dropout(0.4),
    layers.Dense(len(CLASSES),activation='softmax')])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cb=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=6,restore_best_weights=True)]
model.fit(tr, validation_data=va, epochs=30, callbacks=cb)
print('✅ Train xong. Class order:', CLASSES)

In [ ]:
# 5 — Xuất TFLite + tải về
OUT = 'drowsiness_model.tflite' if MODE=='eye' else 'yawn_model.tflite'
conv = tf.lite.TFLiteConverter.from_keras_model(model)
open(OUT,'wb').write(conv.convert())
kb=os.path.getsize(OUT)//1024

# verify
it=tf.lite.Interpreter(model_path=OUT); it.allocate_tensors()
inp=it.get_input_details()[0]
it.set_tensor(inp['index'], np.random.rand(1,64,64,3).astype('float32')); it.invoke()
o=it.get_tensor(it.get_output_details()[0]['index'])[0]
print(f'✅ {OUT} ({kb}KB) | input {inp["shape"]} | output {o} (tổng≈{o.sum():.2f})')

from google.colab import files
files.download(OUT)
print(f'\n📥 Tải {OUT} về → bỏ vào app/src/main/assets/ → build lại APK')

## ✅ Sau khi tải TFLite
1. Copy `drowsiness_model.tflite` (hoặc `yawn_model.tflite`) đè vào `app/src/main/assets/`
2. Build lại APK → cài điện thoại (xem `GUIDE_3H.md`)

**Mẹo nhanh (3h):** quay mỗi class 1-2 video ~10-20 giây, FPS=3 → ~50-150 ảnh/class là đủ demo. Class càng nhiều ảnh + đa dạng (sáng/tối, có kính) → càng tốt.